In [ ]:
import datetime
import logging

import pandas as pd
import pyarrow 
import requests


# Константы и параметры

START_DATE = "2026-01-01"
END_DATE = "2026-02-14"

LATITUDE = 55.7522
LONGITUDE = 37.6156
TIMEZONE = "Europe/Moscow"

BASE_URL = "https://api.open-meteo.com/v1/forecast"

HOURLY_PARAMS = [
    "temperature_2m",
    "snowfall",
    "wind_speed_10m",
    "visibility",
    "rain",
]


# Логирование

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("weather_loader")


# Основная логика

def main() -> None:
    update_at = datetime.date.today()

    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": HOURLY_PARAMS,
        "timezone": TIMEZONE,
    }

    logger.info("Отправка запроса к Open-Meteo API")

    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    logger.info("Сохраним данные в csv и parquet")

    df = pd.DataFrame(data["hourly"])
    df["business_date"] = f"{START_DATE} - {END_DATE}"
    df["update_at"] = update_at


    csv_path = f"moscow_weather_{update_at}.csv"
    parquet_path = f"moscow_weather_{update_at}.parquet"

    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)

    logger.info("Программа выполнена успешно")


# Точка входа

if __name__ == "__main__":
    main()


2026-02-07 16:33:17,827 - INFO - Отправка запроса к Open-Meteo API
2026-02-07 16:33:18,041 - INFO - Сохраним данные в csv и parquet
2026-02-07 16:33:18,047 - INFO - Программа выполнена успешно
